# Geographic-Scale results

Cross-k scaling panels aggregated over models, multipliers and samples, then a per-run drill-down rendered from saved `results.json` and `.npz`. No retraining.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os
import pyrootutils

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator='pyproject.toml',
    pythonpath=True,
    dotenv=True,
)

## Parameters

In [ ]:
from building.models import BASE_LIST

COLLECTION = "scale_49_2_r50"
MODELS = BASE_LIST  # only these (build_model, multiplier) pairs are shown

TASK_SLUG = COLLECTION
TASK_DIR = ROOT / "models" / TASK_SLUG
print(f"task: {TASK_SLUG}   dir: {TASK_DIR}  exists={TASK_DIR.exists()}")


## Cross-k summary plot

In [ ]:
from building.polychirp_training.results import gather_results

df = gather_results(TASK_DIR)
df = df[df["variant"] == "standard"].copy()
keep = {(m.name, m.multi) for m in MODELS}
df = df[[(b, mult) in keep for b, mult in zip(df["build_model"], df["multiplier"])]].copy()
df["k"] = df["k"].astype(int)
df["run"] = df["build_model"].astype(str) + " x" + df["multiplier"].map(lambda m: f"{m:g}")
runs = sorted(df["run"].unique())
collapse = {m.name for m in MODELS if m.base_model.quantize_collapse}  # dropped from quantized graphs, kept in float ones
print(f"{len(df)} runs | {df['run'].nunique()} model x mult | k={sorted(df['k'].unique())}")

ks = sorted(df["k"].unique())
samples = sorted(df["sample_idx"].unique())
counts = df.groupby(["sample_idx", "k"])["run"].nunique().unstack(fill_value=0).reindex(index=samples, columns=ks, fill_value=0)
print("sample\\k " + "".join(f"{k:>4d}" for k in ks))
for s in samples:
    print(f"{s:>7d}  " + "".join(f"{counts.loc[s, k]:>4d}" for k in ks))


In [ ]:
from building.polychirp_training.results import plot_scaling_panels

panels = [("macro_f2", "Macro F2"), ("macro_f1", "Macro F1"), ("macro_recall", "Macro recall"),
          ("macro_precision", "Macro precision"), ("test_loss", "Quant loss")]
plot_scaling_panels(df[~df["build_model"].isin(collapse)], panels, f"Scaling by k: {TASK_SLUG}")


## Float scaling by k

Same panels for the float (pre-quantization) model.

In [ ]:
panels = [("float_macro_f2", "Macro F2"), ("float_macro_f1", "Macro F1"), ("float_macro_recall", "Macro recall"),
          ("float_macro_precision", "Macro precision"), ("test_loss_float", "Float loss")]
plot_scaling_panels(df, panels, f"Float scaling by k: {TASK_SLUG}")


## Quantization drop vs k

One panel per macro metric. x = k, one line per model. Values are INT8 minus float averaged over samples, so a negative line means quality fell.

In [ ]:
import matplotlib.pyplot as plt

drop_metrics = ['precision', 'recall', 'f1', 'f2', 'auc']
dfq = df[~df["build_model"].isin(collapse)]
drops_df = dfq[['run', 'k']].copy()
for m in drop_metrics:
    drops_df[m] = dfq[f'macro_{m}'] - dfq[f'float_macro_{m}']
drop_summary = drops_df.groupby(['run', 'k']).mean().sort_index()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
axes[-1].set_visible(False)
for ax, m in zip(axes, drop_metrics):
    for run in runs:
        if run not in drop_summary.index.get_level_values(0):
            continue
        s = drop_summary.loc[run, m]
        ax.plot(s.index, s.values, marker='o', label=run)
    ax.axhline(0, color='k', lw=0.5)
    ax.set_title(f'macro {m}')
    ax.set_xlabel('k (target classes)')
    ax.set_ylabel('INT8 minus float')
    ax.grid(True, ls='--', alpha=0.5)
    ax.legend(fontsize=8)
fig.suptitle(f'Quantization drop by k: {TASK_SLUG}', fontweight='bold')
plt.tight_layout(); plt.show()


## Export for the paper

Freeze every panel to `<task>/final_results/`: `scaling.json` (mean + std per run and k), plus a pgfplots-ready CSV and a standalone PDF per metric (one clean plot each, not the 6-up grid). Zip that folder for the paper machine.

In [ ]:
from building.polychirp_training.export import export_scaling

# keep / collapse come from the cross-k summary cell above.
export_scaling(TASK_DIR, keep, collapse)


# Per-run drill-down

Pick one grid cell and re-render its per-species block from saved results. No retraining.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from building.polychirp_training.results import load_run
from building.polychirp_training.runner import scale_run_paths
from building.polychirp_training.training import TinyChirpModel
from building.models import model_eval as M

PICK_MODEL = "mel_cnn"
PICK_MULTIPLIER = 1
PICK_K = 3
PICK_SAMPLE = 0
PICK_SPLIT = "test"     # 'train' (train+val) or 'test'
PICK_OBJ = "f2"         # 'f1' or 'f2'

paths = scale_run_paths(TASK_SLUG, TinyChirpModel(base_model=PICK_MODEL, multi=PICK_MULTIPLIER), PICK_K, PICK_SAMPLE)
loaded = load_run(paths.results_path)
assert loaded is not None, f"no results.json at {paths.results_path}"
record = loaded.record
cmp = loaded.cmp(PICK_SPLIT, PICK_OBJ)
float_eval = loaded.eval("keras", PICK_SPLIT, PICK_OBJ)
quant_eval = loaded.eval("tflite", PICK_SPLIT, PICK_OBJ)
label_names = loaded.label_names
print(f"picked: {PICK_MODEL} x{PICK_MULTIPLIER}  k={PICK_K} sample={PICK_SAMPLE}  split={PICK_SPLIT}  thr-opt={PICK_OBJ.upper()}")
print(f"labels: {label_names}")
M.print_summary(quant_eval)


### Per-species table (quantized)

In [ ]:
tbl = M.per_class_table(quant_eval)
display(tbl.style.format(precision=4))
print(
    f'Macro: precision={quant_eval.macro_precision:.4f}  '
    f'recall={quant_eval.macro_recall:.4f}  '
    f'F1={quant_eval.macro_f1:.4f}  '
    f'F2={quant_eval.macro_f2:.4f}  '
    f'AUC={quant_eval.macro_auc:.4f}'
)


### Metric-vs-threshold sweep (quantized)

In [ ]:
M.plot_metric_sweep_panel(quant_eval); plt.show()

### ROC per species (quantized)

In [ ]:
M.plot_roc(quant_eval); plt.show()
for name, mclass in quant_eval.per_class.items():
    print(f'  {name:<28s} AUC={mclass.auc:.4f}' if mclass.auc is not None else f'  {name:<28s} n/a')

### Float vs INT8 comparison

In [ ]:
display(M.comparison_table(cmp).style.set_caption('Float vs INT8 per-class'))
M.plot_quantization_drop(cmp, metric='f2'); plt.show()
M.print_comparison(cmp)

## All measurements: 8 tables per model

(float | INT8) x (train | test) x (F1-opt | F2-opt threshold). Thresholds tuned on train+val per backend.

In [ ]:
M.display_opti_tables(record)

### Confusion matrices (top-1)

In [ ]:
M.plot_confusion_matrices(record.confusion, label_names)
plt.show()